# 🧩 Mini-Lab: Batch Processing & Reporting

**Module 3: Structured Extraction** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Process** multiple documents efficiently
2. **Use** async for parallel processing
3. **Track** progress and handle errors in batches
4. **Generate** summary reports from extractions

## Target Concepts

| Concept | Description |
|---------|-------------|
| Batch processing | Processing multiple documents sequentially or in parallel |
| Async/await | Python's asynchronous programming for concurrent API calls |
| Progress tracking | Using progress bars and status updates for long-running operations |
| Reporting | Generating summary analytics and exports from extracted data |

In [13]:
import json
import asyncio
from pathlib import Path
from datetime import datetime
from openai import OpenAI, AsyncOpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
from IPython.display import Markdown, display
from tqdm.notebook import tqdm

from document_schemas import Invoice, Receipt, Resume, Contract, DOCUMENT_SCHEMAS
from extractor import DocumentExtractor, BatchExtractor

def md(text):
    display(Markdown(text))

load_dotenv()
client = OpenAI()

DATA_DIR = Path('../data')

print('✓ Setup complete')

✓ Setup complete


---
## 1. Load All Documents

Let's load all documents for batch processing.

In [5]:
def load_all_documents() -> dict[str, dict[str, str]]:
    """
    Load all documents organized by type.
    
    Returns:
        Dict of {doc_type: {filename: content}}
    """
    documents = {}
    
    folders = {
        'invoice': 'invoices',
        'receipt': 'receipts',
        'resume': 'resumes',
        'contract': 'contracts'
    }
    
    for doc_type, folder in folders.items():
        folder_path = DATA_DIR / folder
        if folder_path.exists():
            documents[doc_type] = {}
            for file in sorted(folder_path.glob('*.txt')):
                documents[doc_type][file.stem] = file.read_text(encoding='utf-8')
    
    return documents


all_docs = load_all_documents()

print('📁 Loaded documents:')
total = 0
for doc_type, docs in all_docs.items():
    print(f'  {doc_type}: {len(docs)} files')
    total += len(docs)
print(f'\n  Total: {total} documents')

📁 Loaded documents:
  invoice: 6 files
  receipt: 5 files
  resume: 5 files
  contract: 5 files

  Total: 21 documents


---
## 2. Sequential Batch Processing

Process documents one at a time with progress tracking.

In [6]:
def process_batch_sequential(
    documents: list[tuple[str, str, str]],  # (name, content, doc_type)
    extractor: DocumentExtractor
) -> list[dict]:
    """
    Process documents sequentially with progress bar.
    """
    results = []
    
    for name, content, doc_type in tqdm(documents, desc='Extracting'):
        schema = DOCUMENT_SCHEMAS[doc_type]
        
        try:
            extracted = extractor.extract(content, schema)
            results.append({
                'file': name,
                'type': doc_type,
                'success': True,
                'data': extracted,
                'error': None
            })
        except Exception as e:
            results.append({
                'file': name,
                'type': doc_type,
                'success': False,
                'data': None,
                'error': str(e)
            })
    
    return results


# Prepare documents list
doc_list = []
for doc_type, docs in all_docs.items():
    for name, content in docs.items():
        doc_list.append((name, content, doc_type))

# Create extractor
extractor = DocumentExtractor(client)

# Process (limit to 4 for demo speed)
print('🔄 Processing documents sequentially...')
results = process_batch_sequential(doc_list[:4], extractor)

# Summary
success = sum(1 for r in results if r['success'])
print(f'\n✓ Processed {len(results)} documents')
print(f'  Success: {success}/{len(results)}')

🔄 Processing documents sequentially...


Extracting:   0%|          | 0/4 [00:00<?, ?it/s]


✓ Processed 4 documents
  Success: 4/4


---
## 3. Async Batch Processing

Process multiple documents in parallel for better throughput.

In [7]:
async def extract_one_async(
    client: AsyncOpenAI,
    name: str,
    content: str,
    schema: type[BaseModel],
    doc_type: str,
    semaphore: asyncio.Semaphore
) -> dict:
    """
    Extract a single document asynchronously.
    """
    async with semaphore:  # Limit concurrent requests
        try:
            schema_json = json.dumps(schema.model_json_schema(), indent=2)
            
            response = await client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[
                    {
                        'role': 'system',
                        'content': f'Extract data according to schema:\n{schema_json}\nReturn valid JSON.'
                    },
                    {'role': 'user', 'content': f'Document:\n{content}'}
                ],
                response_format={'type': 'json_object'},
                temperature=0
            )
            
            data = json.loads(response.choices[0].message.content)
            validated = schema(**data)
            
            return {
                'file': name,
                'type': doc_type,
                'success': True,
                'data': validated,
                'error': None
            }
            
        except Exception as e:
            return {
                'file': name,
                'type': doc_type,
                'success': False,
                'data': None,
                'error': str(e)
            }


async def process_batch_async(
    documents: list[tuple[str, str, str]],
    max_concurrent: int = 5
) -> list[dict]:
    """
    Process documents in parallel with concurrency limit.
    """
    async_client = AsyncOpenAI()
    semaphore = asyncio.Semaphore(max_concurrent)
    
    tasks = []
    for name, content, doc_type in documents:
        schema = DOCUMENT_SCHEMAS[doc_type]
        task = extract_one_async(
            async_client, name, content, schema, doc_type, semaphore
        )
        tasks.append(task)
    
    # Run all tasks with progress
    results = []
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc='Extracting'):
        result = await coro
        results.append(result)
    
    return results


# Process async (limit to 4 for demo)
print('🔄 Processing documents in parallel...')
async_results = await process_batch_async(doc_list[:4], max_concurrent=3)

success = sum(1 for r in async_results if r['success'])
print(f'\n✓ Processed {len(async_results)} documents')
print(f'  Success: {success}/{len(async_results)}')

🔄 Processing documents in parallel...


Extracting:   0%|          | 0/4 [00:00<?, ?it/s]


✓ Processed 4 documents
  Success: 4/4


---
## 4. Process All Documents

Now let's process all documents and collect results.

In [8]:
# Process all documents
print(f'🔄 Processing all {len(doc_list)} documents...')
print('(This may take a minute)\n')

all_results = await process_batch_async(doc_list, max_concurrent=5)

# Summary
success = sum(1 for r in all_results if r['success'])
failed = len(all_results) - success

print(f'\n✓ Processing complete!')
print(f'  Total: {len(all_results)}')
print(f'  Success: {success}')
print(f'  Failed: {failed}')

🔄 Processing all 21 documents...
(This may take a minute)



Extracting:   0%|          | 0/21 [00:00<?, ?it/s]


✓ Processing complete!
  Total: 21
  Success: 21
  Failed: 0


---
## 5. Generate Reports

Create summary reports from the extracted data.

In [9]:
def generate_report(results: list[dict]) -> str:
    """
    Generate a markdown report from extraction results.
    """
    report = []
    report.append('# Document Extraction Report')
    report.append(f'\n**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    
    # Overall stats
    total = len(results)
    success = sum(1 for r in results if r['success'])
    report.append(f'\n## Summary')
    report.append(f'- **Total Documents:** {total}')
    report.append(f'- **Successful:** {success} ({100*success/total:.1f}%)')
    report.append(f'- **Failed:** {total - success}')
    
    # By type
    report.append(f'\n## Results by Type\n')
    
    for doc_type in ['invoice', 'receipt', 'resume', 'contract']:
        type_results = [r for r in results if r['type'] == doc_type]
        if not type_results:
            continue
        
        type_success = sum(1 for r in type_results if r['success'])
        report.append(f'### {doc_type.title()}s ({type_success}/{len(type_results)})')
        
        for r in type_results:
            status = '✅' if r['success'] else '❌'
            
            if r['success'] and r['data']:
                data = r['data']
                
                if doc_type == 'invoice':
                    info = f"{data.vendor_name} → ${data.total:,.2f}"
                elif doc_type == 'receipt':
                    info = f"{data.store_name} → ${data.total:,.2f}"
                elif doc_type == 'resume':
                    info = f"{data.full_name} ({data.email})"
                else:  # contract
                    info = f"{data.contract_title}"
                
                report.append(f'- {status} **{r["file"]}**: {info}')
            else:
                report.append(f'- {status} **{r["file"]}**: {r["error"][:50]}...')
        
        report.append('')
    
    return '\n'.join(report)


# Generate and display report
report = generate_report(all_results)
md(report)

# Document Extraction Report

**Generated:** 2026-03-28 20:21:31

## Summary
- **Total Documents:** 21
- **Successful:** 21 (100.0%)
- **Failed:** 0

## Results by Type

### Invoices (6/6)
- ✅ **invoice_003_email_style**: DesignPro Studio → $15,000.00
- ✅ **invoice_004_european_format**: Müller & Schmidt GmbH → $11,959.50
- ✅ **invoice_006_minimal**: QuickShip Logistics → $65.00
- ✅ **invoice_001_corporate**: TechCorp Solutions Inc. → $24,686.46
- ✅ **invoice_005_handwritten_style**: Pete's Plumbing Services → $355.00
- ✅ **invoice_002_simple_receipt**: CloudHost Services → $406.00

### Receipts (5/5)
- ✅ **receipt_003_grocery**: SAFEWAY SUPERMARKET → $103.08
- ✅ **receipt_002_restaurant**: THE GARDEN BISTRO → $162.33
- ✅ **receipt_004_gas_station**: SHELL SERVICE STATION #2847 → $65.25
- ✅ **receipt_001_retail**: BEST BUY ELECTRONICS → $2,667.95
- ✅ **receipt_005_online**: Amazon.com → $145.35

### Resumes (5/5)
- ✅ **resume_002_recent_graduate**: JESSICA MARTINEZ (jessica.martinez@email.com)
- ✅ **resume_001_software_engineer**: ALEXANDER CHEN (alex.chen@email.com)
- ✅ **resume_003_executive**: MICHAEL ROBERTSON (m.robertson@email.com)
- ✅ **resume_004_career_changer**: DAVID KIM (david.kim@email.com)
- ✅ **resume_005_freelancer**: SARAH PATEL (sarah.patel@email.com)

### Contracts (5/5)
- ✅ **contract_003_nda**: Mutual Non-Disclosure Agreement
- ✅ **contract_004_lease**: Commercial Office Space Lease
- ✅ **contract_005_freelance**: Web Development Services - Independent Contractor
- ✅ **contract_001_employment**: Senior Software Engineer Employment Agreement
- ✅ **contract_002_service**: Custom Web Application Development Services


---
## 6. Export Results

Save extracted data for further use.

In [10]:
def export_results(results: list[dict], output_dir: Path):
    """
    Export results to JSON files.
    """
    output_dir.mkdir(exist_ok=True)
    
    exported = 0
    
    for r in results:
        if not r['success'] or not r['data']:
            continue
        
        # Convert Pydantic model to dict
        data_dict = r['data'].model_dump()
        
        # Add metadata
        output = {
            'source_file': r['file'],
            'document_type': r['type'],
            'extracted_at': datetime.now().isoformat(),
            'data': data_dict
        }
        
        # Save to JSON
        output_file = output_dir / f"{r['file']}.json"
        with open(output_file, 'w') as f:
            json.dump(output, f, indent=2, default=str)
        
        exported += 1
    
    return exported


# Export results
output_dir = Path('extracted_data')
exported = export_results(all_results, output_dir)

print(f'✓ Exported {exported} files to {output_dir}/')

# Show exported files
print('\nExported files:')
for f in sorted(output_dir.glob('*.json'))[:5]:
    print(f'  - {f.name}')
if len(list(output_dir.glob('*.json'))) > 5:
    print(f'  ... and {len(list(output_dir.glob("*.json"))) - 5} more')

✓ Exported 21 files to extracted_data/

Exported files:
  - contract_001_employment.json
  - contract_002_service.json
  - contract_003_nda.json
  - contract_004_lease.json
  - contract_005_freelance.json
  ... and 16 more


---
## 7. Analytics Dashboard

Quick analytics from the extracted data.

In [11]:
def generate_analytics(results: list[dict]):
    """
    Generate analytics from extraction results.
    """
    # Invoice analytics
    invoices = [r['data'] for r in results if r['type'] == 'invoice' and r['success']]
    if invoices:
        total_value = sum(inv.total for inv in invoices)
        avg_value = total_value / len(invoices)
        print('📊 INVOICE ANALYTICS')
        print(f'   Total invoices: {len(invoices)}')
        print(f'   Total value: ${total_value:,.2f}')
        print(f'   Average value: ${avg_value:,.2f}')
        print()
    
    # Receipt analytics
    receipts = [r['data'] for r in results if r['type'] == 'receipt' and r['success']]
    if receipts:
        total_spent = sum(rec.total for rec in receipts)
        stores = set(rec.store_name for rec in receipts)
        print('📊 RECEIPT ANALYTICS')
        print(f'   Total receipts: {len(receipts)}')
        print(f'   Total spent: ${total_spent:,.2f}')
        print(f'   Unique stores: {len(stores)}')
        print()
    
    # Resume analytics
    resumes = [r['data'] for r in results if r['type'] == 'resume' and r['success']]
    if resumes:
        all_skills = []
        for res in resumes:
            all_skills.extend(res.skills)
        
        # Top skills
        from collections import Counter
        skill_counts = Counter(all_skills)
        
        print('📊 RESUME ANALYTICS')
        print(f'   Total resumes: {len(resumes)}')
        print(f'   Unique skills: {len(set(all_skills))}')
        print(f'   Top skills:')
        for skill, count in skill_counts.most_common(5):
            print(f'      - {skill}: {count}')
        print()
    
    # Contract analytics
    contracts = [r['data'] for r in results if r['type'] == 'contract' and r['success']]
    if contracts:
        total_value = sum(c.total_value for c in contracts if c.total_value)
        types = set(c.contract_type for c in contracts)
        print('📊 CONTRACT ANALYTICS')
        print(f'   Total contracts: {len(contracts)}')
        print(f'   Total value: ${total_value:,.2f}')
        print(f'   Contract types: {types}')


# Generate analytics
generate_analytics(all_results)

📊 INVOICE ANALYTICS
   Total invoices: 6
   Total value: $52,471.96
   Average value: $8,745.33

📊 RECEIPT ANALYTICS
   Total receipts: 5
   Total spent: $3,143.96
   Unique stores: 5

📊 RESUME ANALYTICS
   Total resumes: 5
   Unique skills: 64
   Top skills:
      - Python: 5
      - React: 5
      - Node.js: 5
      - MongoDB: 5
      - PostgreSQL: 5

📊 CONTRACT ANALYTICS
   Total contracts: 5
   Total value: $535,000.00
   Contract types: {'Service', 'Lease', 'Employment', 'NDA', 'Freelance'}


## 🎯 Summary

### Key Takeaways

1. **Sequential processing** — simple loop with progress tracking for small batches
2. **Async processing** — use `asyncio` and `AsyncOpenAI` for parallel API calls with concurrency limits
3. **Progress tracking** — `tqdm` provides visual feedback for long-running batch operations
4. **Export and reporting** — convert extracted data to JSON files and generate analytics dashboards
5. **Semaphores** — control concurrent API requests to avoid rate limits and manage resource usage

---
## 8. Project Complete!

### What We Built

1. **Basic Extraction** (Notebook 1)
   - Zero-shot and few-shot extraction
   - Output parsing

2. **Schema Validation** (Notebook 2)
   - JSON mode
   - Pydantic models
   - Nested structures

3. **Multi-Document Types** (Notebook 3)
   - Document classification
   - Universal extractor

4. **Error Handling** (Notebook 4)
   - Confidence scoring with logprobs
   - Retry strategies
   - Robust extraction

5. **Batch Processing** (Notebook 5)
   - Parallel processing
   - Progress tracking
   - Reporting and export

---
## Cleanup

In [12]:
# Optional: Clean up exported files
# import shutil
# shutil.rmtree('extracted_data', ignore_errors=True)

print('🎉 Project 1: Document Extractor - Complete!')

🎉 Project 1: Document Extractor - Complete!
